# ddqn-router — Quickstart

This notebook walks you through the full ddqn-router workflow:
1. Define your agents
2. Use a built-in demo dataset (no API key needed)
3. Train the DDQN router
4. Route queries and inspect decisions

**Run all cells top to bottom.** Total time: ~5 minutes.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kirmoz1997/ddqn-router/blob/main/examples/quickstart.ipynb)

In [ ]:
# Install ddqn-router (skip if already installed)
!pip install ddqn-router -q

## Step 1 — Define your agents

Every ddqn-router project starts with a YAML config that lists your agents.
Each agent has an `id`, a `name`, and a `description` that tells the system what it handles.

In [ ]:
import os

config_yaml = """
agents:
  - id: 0
    name: "Billing Agent"
    description: "Handles billing, invoices, payments, refunds, subscriptions"
  - id: 1
    name: "Technical Agent"
    description: "Handles bugs, errors, crashes, API issues, integrations"
  - id: 2
    name: "Account Agent"
    description: "Handles account settings, passwords, access, permissions"

training:
  total_steps: 30000
  val_eval_freq: 2000

dataset:
  input: "./data/tasks.jsonl"

output_dir: "./quickstart_artifacts/"
""".strip()

with open("config.yaml", "w") as f:
    f.write(config_yaml)

print("config.yaml written")
print(config_yaml)

## Step 2 — Dataset

Normally you would label your own queries using `ddqn-router label` with an LLM.
For this quickstart, we use a built-in demo dataset of 50 labeled examples so
you can see the full training cycle immediately — **no API key needed**.

In [ ]:
import json

os.makedirs("data", exist_ok=True)

demo_tasks = [
    # [0] Billing only
    {"text": "I was charged twice for my monthly subscription", "required_agents": [0]},
    {"text": "Where can I find my latest invoice?", "required_agents": [0]},
    {"text": "I need a refund for the accidental purchase", "required_agents": [0]},
    {"text": "How do I update my credit card on file?", "required_agents": [0]},
    {"text": "My promo code is not applying the discount", "required_agents": [0]},
    {"text": "Can I switch from annual to monthly billing?", "required_agents": [0]},
    {"text": "I was billed after I cancelled my plan", "required_agents": [0]},
    {"text": "The payment failed but money was deducted", "required_agents": [0]},
    # [1] Technical only
    {"text": "The API keeps returning a 500 error on POST requests", "required_agents": [1]},
    {"text": "Webhook events are not being delivered", "required_agents": [1]},
    {"text": "The dashboard crashes when I open the reports tab", "required_agents": [1]},
    {"text": "I am getting a timeout when uploading large files", "required_agents": [1]},
    {"text": "The SDK throws a null pointer exception on init", "required_agents": [1]},
    {"text": "Search is returning incorrect results", "required_agents": [1]},
    {"text": "The mobile app freezes after the latest update", "required_agents": [1]},
    {"text": "Integration with Slack is broken since yesterday", "required_agents": [1]},
    # [2] Account only
    {"text": "I forgot my password and the reset email never arrived", "required_agents": [2]},
    {"text": "How do I enable two-factor authentication?", "required_agents": [2]},
    {"text": "I need to change the email address on my account", "required_agents": [2]},
    {"text": "Can I transfer ownership of my organization?", "required_agents": [2]},
    {"text": "A team member needs admin permissions", "required_agents": [2]},
    {"text": "My account was locked after too many login attempts", "required_agents": [2]},
    {"text": "How do I delete my account permanently?", "required_agents": [2]},
    {"text": "I want to merge two accounts into one", "required_agents": [2]},
    # [0, 2] Billing + Account
    {"text": "The wrong account was charged for the subscription", "required_agents": [0, 2]},
    {"text": "I need a refund and then close my account", "required_agents": [0, 2]},
    {
        "text": "Billing is going to an old email I no longer have access to",
        "required_agents": [0, 2],
    },
    {"text": "Transfer the subscription to a different account", "required_agents": [0, 2]},
    {"text": "My team plan is charged to a personal account by mistake", "required_agents": [0, 2]},
    {
        "text": "I updated my payment method but the old card is still being charged",
        "required_agents": [0, 2],
    },
    {"text": "Downgrade my plan and update the billing contact", "required_agents": [0, 2]},
    {
        "text": "I left the company but still get charged on my personal card",
        "required_agents": [0, 2],
    },
    # [1, 2] Technical + Account
    {"text": "I cannot log in via the API — it says access denied", "required_agents": [1, 2]},
    {"text": "SSO integration is not recognizing my account", "required_agents": [1, 2]},
    {"text": "My API key stopped working after I changed my password", "required_agents": [1, 2]},
    {"text": "OAuth callback returns an invalid token for my user", "required_agents": [1, 2]},
    {"text": "Two-factor auth is blocking the CLI tool from connecting", "required_agents": [1, 2]},
    {
        "text": "Permission denied error when accessing the admin API endpoint",
        "required_agents": [1, 2],
    },
    {"text": "The webhook secret rotated and now my integration fails", "required_agents": [1, 2]},
    {
        "text": "I cannot accept the team invite — the link gives a 403 error",
        "required_agents": [1, 2],
    },
    # [0, 1, 2] All three
    {
        "text": "Cancel my account, refund the last charge, and revoke all API keys",
        "required_agents": [0, 1, 2],
    },
    {
        "text": "I was double-charged and now the dashboard shows an error on my account page",
        "required_agents": [0, 1, 2],
    },
    {
        "text": "Migrate my billing to a new account and fix the broken webhook integration",
        "required_agents": [0, 1, 2],
    },
    {
        "text": "My payment failed, the app crashed, and now I am locked out of my account",
        "required_agents": [0, 1, 2],
    },
    {
        "text": "Refund the overcharge, reset my password, and fix the API returning 401",
        "required_agents": [0, 1, 2],
    },
    {
        "text": "Update billing info, change account owner, and re-authenticate the Slack integration",
        "required_agents": [0, 1, 2],
    },
    {
        "text": "I need an invoice correction, my SSO is broken, and account email needs updating",
        "required_agents": [0, 1, 2],
    },
    {
        "text": "Subscription charge went to wrong card, account is locked, and API keys expired",
        "required_agents": [0, 1, 2],
    },
    {
        "text": "Close account, get final invoice, and export all data via the API before deletion",
        "required_agents": [0, 1, 2],
    },
    {
        "text": "Payment dispute, need account ownership transfer, and the webhook endpoint is down",
        "required_agents": [0, 1, 2],
    },
]

with open("data/tasks.jsonl", "w") as f:
    for i, task in enumerate(demo_tasks):
        task["id"] = f"ex_{i:04d}"
        f.write(json.dumps(task) + "\n")

print(f"Created {len(demo_tasks)} labeled examples in data/tasks.jsonl")

In [ ]:
import subprocess

subprocess.run(["ddqn-router", "dataset", "stats", "--input", "data/tasks.jsonl"])
subprocess.run(["ddqn-router", "dataset", "split", "--input", "data/tasks.jsonl"])

## Step 3 — Train the DDQN router

We train for 30,000 steps (reduced for demo speed — takes ~2 minutes on CPU).
For production use, the default 200,000 steps gives significantly better results.

In [ ]:
import subprocess

result = subprocess.run(["ddqn-router", "train", "--config", "config.yaml"], capture_output=False)

In [ ]:
import json

with open("quickstart_artifacts/metrics_test.json") as f:
    metrics = json.load(f)

print("=== Test Metrics ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

## Step 4 — Route queries

Load the trained router and try it on some example queries.

In [ ]:
from ddqn_router import DDQNRouter

router = DDQNRouter.load("./quickstart_artifacts/")

test_queries = [
    "I was charged twice for my subscription",
    "The API keeps returning a 500 error",
    "I can't reset my password",
    "Wrong card was charged and I also can't log in",
    "Cancel my account and issue a full refund for the last payment",
]

print("=== Routing Results ===\n")
for query in test_queries:
    result = router.route(query)
    print(f"Query:  {query}")
    print(f"Agents: {result.agent_names}  (confidence: {result.confidence:.2f})\n")

In [ ]:
print("=== Step-by-step explanation ===\n")
router.explain("Cancel my account and issue a full refund for the last payment")

## Step 5 — Label your own data (optional)

The built-in demo dataset above is small and synthetic. For production use:

1. Collect representative queries from your actual system (500–2000 is ideal)
2. Save them to a text file (one query per line)
3. Use `ddqn-router label` to annotate them automatically with an LLM

The labeler works with any OpenAI-compatible API — OpenAI, DeepSeek, Ollama, etc.

In [ ]:
# OPTIONAL: Run this cell only if you have an API key and your own queries file

import os

os.environ["DDQN_ROUTER_API_KEY"] = "sk-..."  # replace with your key

# Create a sample input file
with open("my_queries.txt", "w") as f:
    f.write("I need to cancel my plan and get a refund\n")
    f.write("The webhook is not firing on payment events\n")
    f.write("How do I add a new team member to my account?\n")

!ddqn-router label \
    --config config.yaml \
    --input my_queries.txt \
    --output data/my_labeled.jsonl \
    --model gpt-4o-mini

## Next steps

- **More data** — label 500–2000 queries for production-quality routing
- **More agents** — add agents to `config.yaml` and retrain
- **Serve as API** — `pip install ddqn-router[serve]` then `ddqn-router serve`
- **Integrate** — `from ddqn_router import DDQNRouter` in your MAS orchestration code

Full documentation: https://github.com/kirmoz1997/ddqn-router